# Creating an Arbitrary Index from Nasdaq Company Tickers

The goal of this project is to create an arbitrary stock index based on Nasdaq-listed companies whose ticker symbols follow a specific rule.

For example, the **Y Index** contains all eligible Nasdaq-listed companies whose ticker starts with the letter **Y**.

The objective is not only to create the index, but also to build a reproducible pipeline that:

1. Downloads the universe of Nasdaq-listed securities.
2. Identifies real operating companies.
3. Removes financial instruments that are not suitable for an equity index.
4. Applies eligibility rules similar to professional index providers.
5. Calculates index constituents and weights.
6. Performs exploratory data analysis (EDA) on the resulting index.

---

## 1. Data Sources

### SEC Company Tickers

The SEC provides a list of companies that file reports with the U.S. Securities and Exchange Commission.

Source:

https://www.sec.gov/files/company_tickers.json

This dataset contains:

- Company name
- Ticker symbol
- SEC CIK identifier

Example:

```json
{
"ticker": "AAPL",
"title": "Apple Inc.",
"cik_str": 320193
}
```

The SEC dataset is useful because it represents **actual reporting companies**, unlike exchange symbol lists that may contain many financial products.

---

### Nasdaq Listed Securities

Source:

https://ftp.nasdaqtrader.com/SymbolDirectory/nasdaqlisted.txt

This file contains all securities listed on Nasdaq.

It includes:

- Common stocks
- ADRs
- ETFs
- Warrants
- Rights
- Units
- Preferred shares
- Other financial instruments

The Nasdaq dataset is required because it provides the official universe of securities traded on Nasdaq.

### Data location

Both files are in the `/data` directory.

---

## 2. Building the Company Universe

We combine the SEC and Nasdaq datasets to create a clean universe of companies.

The process is:


Nasdaq Listed Securities --> Remove ETFs and test securities --> Join with SEC reporting companies -- >
Remove non-equity securities --> Remove SPACs --> Apply company age filters (remove less than 1 year)

This gives us the universe where we can select our companies for the index.

---

## 3. Security Filtering Rules

We exclude the following securities because they do not represent ordinary company ownership.

- ETFs
- Test issues
- NextShares
- Warrants
- Rights
- Units
- Preferred shares
- Bonds and notes
- Other derivative instruments

Examples:

| Symbol | Type | Action |
|---|---|---|
| AAPL | Common stock | Keep |
| YICC | Common stock | Keep |
| YICCU | Unit | Remove |
| YICCW | Warrant | Remove |

---

## 4. Company Eligibility Rules

We apply other filters to create a more realistic index universe:

### Remove SPACs

We remove Special Purpose Acquisition Companies.

Examples of excluded names:

- Acquisition Corp
- Acquisition Holdings
- Blank Check Company

The reason is that these entities do not represent mature operating businesses.

---

### Minimum Company Age

We exclude companies younger than one year.

This avoids including:

- Recently created SPACs
- Very recent IPOs
- **Companies without sufficient trading history**

---

## 5. Creating the Letter-Based Index Universe

After cleaning the Nasdaq company universe, we group companies by ticker letter.

- A Index
- J Index
- Y Index
- Any other ticker-based universe

## GOAL: Find the Most Underrepresented Letter in the tickers, and Create an Index Based on it

In [1]:
from src.universe import build_company_universe
from src.index import (
    companies_with_letter,
    ticker_letter_counts,
    least_common_letters,
    build_j_index_constituents,
)

In [2]:
universe = build_company_universe(
    "/home/eric/Documents/mess_box/y_index/data/sec.json",
    "/home/eric/Documents/mess_box/y_index/data/nasdaqlisted.txt",
)

In [3]:
counts = ticker_letter_counts(universe)

print(counts)

least_common_letters(universe)

   letter  companies
0       J         40
1       Q         52
2       Z         91
3       U        177
4       W        178
5       Y        192
6       K        229
7       V        238
8       H        247
9       F        281
10      X        301
11      G        339
12      D        365
13      P        392
14      O        430
15      B        438
16      E        446
17      M        458
18      I        525
19      L        538
20      N        602
21      S        657
22      R        658
23      C        660
24      A        661
25      T        666


,letter,companies
0,J,40
1,Q,52
2,Z,91
3,U,177
4,W,178
5,Y,192
6,K,229
7,V,238
8,H,247
9,F,281


In [4]:
j_index = companies_with_letter(universe, "J")

j_index

,Symbol,Security Name,Market Category,Test Issue,Financial Status,Round Lot Size,ETF,NextShares,CIK,Company
0,AIRJ,AirJoule Technologies Corporation - Class A Co...,S,N,N,100.0,N,N,1855474,AirJoule Technologies Corp.
1,BJDX,"Bluejay Diagnostics, Inc. - Common Stock",S,N,N,100.0,N,N,1704287,"Bluejay Diagnostics, Inc."
2,BJRI,"BJ's Restaurants, Inc. - Common Stock",Q,N,N,100.0,N,N,1013488,BJs RESTAURANTS INC
3,BOTJ,"Bank of the James Financial Group, Inc. - Comm...",S,N,N,100.0,N,N,1275101,BANK OF THE JAMES FINANCIAL GROUP INC
4,CJMB,Callan JMB Inc. - Common Stock,S,N,D,100.0,N,N,2032545,CALLAN JMB INC.
5,DJCO,Daily Journal Corp. (S.C.) - Common Stock,S,N,N,40.0,N,N,783412,DAILY JOURNAL CORP
6,DJT,Trump Media & Technology Group Corp. - Common ...,G,N,N,100.0,N,N,1849635,Trump Media & Technology Group Corp.
7,EJH,E-Home Household Service Holdings Limited - Or...,S,N,D,100.0,N,N,1769768,E-Home Household Service Holdings Ltd
8,INTJ,Intelligent Group Limited - Class A Ordinary S...,S,N,N,100.0,N,N,1916416,Intelligent Group Ltd
9,JACK,Jack In The Box Inc. - Common Stock,Q,N,N,100.0,N,N,807882,JACK IN THE BOX INC


In [5]:
from src.utils import save_parquet_cache

save_parquet_cache(j_index, "data/j_index.parquet")

## Retrieving more data for the J Index

The letter that's more under-represented in the index is the letter J. This allows us to create the J Index, which has 40 companies in it.

Next, we will create a dataframe with some extra information about these companies (market capitalization and size, and sector). With this extra data, we'll be able to do some initial EDA explorations.

In [6]:
j_index_constituents = build_j_index_constituents(j_index)

j_index_constituents

,Symbol,Company,Sector,MarketCap,Weight,Size,Currency,Country,SIC,SIC_Description
0,JKHY,JACK HENRY & ASSOCIATES INC,Technology,10698826752,0.525998,Large Cap,USD,United States,7373,Services-Computer Integrated Systems Design
1,DJT,Trump Media & Technology Group Corp.,Communication Services,2370724864,0.116554,Mid Cap,USD,United States,7370,"Services-Computer Programming, Data Processing..."
2,JJSF,J&J SNACK FOODS CORP,Consumer Defensive,1433511168,0.070477,Small Cap,USD,United States,2052,Cookies & Crackers
3,BJRI,BJs RESTAURANTS INC,Consumer Cyclical,1405191552,0.069085,Small Cap,USD,United States,5812,Retail-Eating Places
4,RJET,REPUBLIC AIRWAYS HOLDINGS INC.,Industrials,809681664,0.039807,Small Cap,USD,United States,4512,"Air Transportation, Scheduled"
5,DJCO,DAILY JOURNAL CORP,Technology,792327872,0.038954,Small Cap,USD,United States,2711,Newspapers: Publishing or Publishing & Printing
6,JOUT,JOHNSON OUTDOORS INC,Consumer Cyclical,481731104,0.023684,Small Cap,USD,United States,3949,"Sporting & Athletic Goods, NEC"
7,JMSB,"John Marshall Bancorp, Inc.",Financial Services,315267072,0.015500,Small Cap,USD,United States,6022,State Commercial Banks
8,AIRJ,AirJoule Technologies Corp.,Industrials,301910464,0.014843,Small Cap,USD,United States,3585,Air-Cond & Warm Air Heatg Equip & Comm & Indl ...
9,JACK,JACK IN THE BOX INC,Consumer Cyclical,277521120,0.013644,Micro Cap,USD,United States,5812,Retail-Eating Places
